In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import pandas as pd
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5" # Set GPU index for local embedding model as needed
os.environ['OPENAI_BASE_URL'] = "http://localhost:6000/v1"  # Update to your OpenAI-compatible server
os.environ['EMB_CACHE_DIR'] = "embedding_caches"
llm_model = "google/medgemma-27b-it"
from hypothesaes.embedding import get_local_chunk_averaged_embeddings

Load All Input Data

In [ ]:
current_dir = os.getcwd()
if current_dir.endswith("notebooks"):
    prefix = "../"
else:
    prefix = "./"

base_dir = os.path.join(prefix, "synthetic_data") #use synthetic data for baselines
train_df = pd.read_json(os.path.join(base_dir, "train_K_5_count_7979.json"), lines=True)
val_df = pd.read_json(os.path.join(base_dir, "validation_K_5_count_2000.json"), lines=True)
test_df = pd.read_json(os.path.join(base_dir, "heldout_K_5_count_2000.json"), lines = True)
texts = train_df["text"].tolist()
labels = train_df["label_synthetic"]
val_texts = val_df["text"].tolist()


In [ ]:
EMBEDDER = "Qwen3-Embedding-0.6B"
CACHE_NAME = f"WIKI-5{EMBEDDER}"
text2embedding = get_local_chunk_averaged_embeddings(texts + val_texts, model="Qwen/Qwen3-Embedding-0.6B", batch_size=8, cache_name=CACHE_NAME, max_tokens = 4000)
embeddings = np.stack([text2embedding[text] for text in texts])
train_embeddings = np.stack([text2embedding[text] for text in texts])
val_embeddings = np.stack([text2embedding[text] for text in val_texts])